# **Point Net - Classification**

In [1]:
import os
import re
from glob import glob
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
from torchmetrics.classification import MulticlassMatthewsCorrCoef
import open3d as o3

from open3d.web_visualizer import draw # for non Colab
import os, os.path as osp

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


In [2]:
NUM_TRAIN_POINTS = 2500
NUM_TEST_POINTS = 10000
NUM_CLASSES = 16
GLOBAL_FEATS = 1024
BATCH_SIZE = 32

            
ROOT = osp.abspath("shapenet_like_out")

In [3]:
from torch.utils.data import DataLoader
from UTILS.foil_dataset import convert_to_shapenet_like

npoints = 10000
convert_to_shapenet_like(root_out = "shapenet_like_out3",npoints=npoints, manifest_keys = ["full_train","full_test"]) # the full train set
# convert_to_shapenet_like(root_out = "shapenet_like_out2", npoints=npoints, manifest_key = "full_test") # the full test set

Converting to ShapeNet-like: 100%|██████████| 1000/1000 [08:46<00:00,  1.90it/s]

[OK] Test set prêt dans: c:\Users\romai\Desktop\3DFundation4SurogateModeling\3DFundation4SurogateModeling2\point_netANDpoint_MAE\shapenet_like_out3
  Catégorie: default -> 00000000
  Exemples (objets): 1000 | npoints/objet: 10000


{'root_out': 'c:\\Users\\romai\\Desktop\\3DFundation4SurogateModeling\\3DFundation4SurogateModeling2\\point_netANDpoint_MAE\\shapenet_like_out3',
 'tokens': ['00000000/airFoil2D_SST_36.622_11.319_3.941_5.424_1.0_16.283',
  '00000000/airFoil2D_SST_58.831_-3.563_2.815_4.916_10.078',
  '00000000/airFoil2D_SST_43.327_8.905_4.236_6.511_10.744',
  '00000000/airFoil2D_SST_89.151_4.462_0.974_4.094_1.0_19.856',
  '00000000/airFoil2D_SST_87.422_2.992_1.895_3.128_1.0_10.656',
  '00000000/airFoil2D_SST_42.531_-3.927_1.264_6.838_0.0_7.489',
  '00000000/airFoil2D_SST_46.246_9.78_3.769_2.353_17.745',
  '00000000/airFoil2D_SST_67.481_-1.259_5.136_2.438_18.882',
  '00000000/airFoil2D_SST_45.701_-1.925_0.671_7.455_0.0_12.285',
  '00000000/airFoil2D_SST_85.488_6.826_3.112_3.445_1.0_17.471',
  '00000000/airFoil2D_SST_56.177_2.108_1.208_3.592_0.0_14.763',
  '00000000/airFoil2D_SST_46.0_-0.095_2.322_1.874_16.58',
  '00000000/airFoil2D_SST_81.846_10.688_0.402_4.721_0.0_11.955',
  '00000000/airFoil2D_SST_56.7

In [8]:
from UTILS.foil_dataset import create_extra_category_from_existing
create_extra_category_from_existing(
    root_out="shapenet_like_out3",
    from_category_id="00000000",
    new_category_id="11111111",
    new_category_name="centered",
    transform_mode="centered"
)



=== Création catégorie centered (11111111) ===


Transforming centered: 100%|██████████| 1000/1000 [00:48<00:00, 20.48it/s]

[OK] Ajouté 1000 tokens dans test split.


In [9]:
from UTILS.foil_dataset import create_extra_category_from_existing
create_extra_category_from_existing(
    root_out="shapenet_like_out3",
    from_category_id="00000000",
    new_category_id="22222222",
    new_category_name="normalized",
    transform_mode="normalized"
)



=== Création catégorie normalized (22222222) ===


Transforming normalized: 100%|██████████| 1000/1000 [01:11<00:00, 14.08it/s]

[OK] Ajouté 1000 tokens dans test split.


In [10]:
from UTILS.foil_dataset import create_extra_category_extruded

create_extra_category_extruded(
    root_out="shapenet_like_out3",
    from_category_id="00000000",
    new_category_id="44444444",
    new_category_name="default_Extruded",
    thickness=1.0,
    k_layers=3
)



=== Création catégorie extrudée default_Extruded (44444444) ===


Extruding default_Extruded: 100%|██████████| 1000/1000 [01:21<00:00, 12.29it/s]

[OK] Ajouté 1000 objets extrudés au test split.


In [11]:
from UTILS.foil_dataset import create_extra_category_extruded
create_extra_category_extruded(
    root_out="shapenet_like_out3",
    from_category_id="11111111",
    new_category_id="55555555",
    new_category_name="centered_Extruded",
    thickness=1.0,
    k_layers=3
)



=== Création catégorie extrudée centered_Extruded (55555555) ===


Extruding centered_Extruded: 100%|██████████| 1000/1000 [01:49<00:00,  9.12it/s]

[OK] Ajouté 1000 objets extrudés au test split.


In [12]:
create_extra_category_extruded(
    root_out="shapenet_like_out3",
    from_category_id="22222222",
    new_category_id="66666666",
    new_category_name="normalized_Extruded",
    thickness=1.0,
    k_layers=3
)


=== Création catégorie extrudée normalized_Extruded (66666666) ===


Extruding normalized_Extruded: 100%|██████████| 1000/1000 [02:16<00:00,  7.32it/s]

[OK] Ajouté 1000 objets extrudés au test split.
